# 04 — Model Evaluation

Visual companion to `python -m src.models.evaluate`. Reads the artifacts
already written by Phase 4 (`models/{interval}/`) and the processed feature/
label tables — **no training or scoring logic lives here**, only plots over
numbers the CLI already computed and validated.

Same honesty rule as the rest of the project: every accuracy number is shown
next to its base rate, and nothing here should be read as "the model works"
without that comparison.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.models.evaluate import base_rate, load_artifacts
from src.models.train import (
    assemble_dataset,
    load_modeling_config,
    make_split_bounds,
    split_dataset,
)

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

In [ ]:
INTERVAL = "1d"

cfg = load_modeling_config("configs/config.yaml")
artifacts = load_artifacts(INTERVAL, cfg)
manifest = artifacts["manifest"]

merged, feature_cols = assemble_dataset(INTERVAL, cfg)
bounds = make_split_bounds(
    len(merged),
    train_frac=cfg["modeling"]["split"]["train_frac"],
    val_frac=cfg["modeling"]["split"]["val_frac"],
    gap_candles=cfg["modeling"]["split"]["gap_candles"],
)
splits = split_dataset(
    merged, bounds, feature_cols=feature_cols, label_col=manifest["label_col"],
)

test = splits["test"]
x_test = test[feature_cols]
y_test = test[manifest["label_col"]].astype("int64").to_numpy()

x_test_scaled = pd.DataFrame(
    artifacts["scaler"].transform(x_test), columns=feature_cols, index=x_test.index
)

prob_lr = artifacts["logistic_regression"].predict_proba(x_test_scaled)[:, 1]
prob_lgb = artifacts["lightgbm"].predict_proba(x_test)[:, 1]
MODELS = (("Logistic Regression", prob_lr), ("LightGBM", prob_lgb))

print(f"{cfg['symbol']} {INTERVAL} — test split: {len(test)} rows, "
      f"{test['open_time'].iloc[0]} -> {test['open_time'].iloc[-1]}")
rate, majority = base_rate(y_test)
print(f"base rate: {rate:.1%} ({majority})")

## 1. Predicted probability distribution

Where does each model's confidence actually sit? A model that never strays
far from 0.5 has little to gate on; a model piling up near 0 and 1 is either
genuinely confident or — on data this hard — a leak warning sign.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (name, prob) in zip(axes, MODELS, strict=True):
    ax.hist(prob, bins=30, range=(0, 1), color="#4C72B0", edgecolor="white")
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    threshold = cfg["modeling"]["confidence_threshold"]
    ax.axvline(threshold, color="#C44E52", linestyle=":", linewidth=1)
    ax.axvline(1 - threshold, color="#C44E52", linestyle=":", linewidth=1)
    ax.set_title(f"{name} — predicted P(up), test split")
    ax.set_xlabel("P(up)")
axes[0].set_ylabel("count")
fig.tight_layout()
plt.show()

## 2. Confusion matrices

Rows are the actual outcome, columns are what the model predicted — the
same matrices `evaluate.py` prints as text, rendered as heatmaps.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, prob) in zip(axes, MODELS, strict=True):
    y_pred = (prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["pred down", "pred up"])
    ax.set_yticks([0, 1], ["actual down", "actual up"])
    ax.set_title(name)
fig.tight_layout()
plt.show()

## 3. Feature importance

Logistic-regression coefficients (scaled-feature space, sign = direction)
and LightGBM gain-based importance, top 15 each. Worth a sanity check: does
anything here look like it shouldn't have predictive power at all — that's
the classic signature of a leaked column.

In [ ]:
from src.models.evaluate import lightgbm_importances, logreg_importances

lr_imp = logreg_importances(artifacts["logistic_regression"], feature_cols).head(15)
lgb_imp = lightgbm_importances(artifacts["lightgbm"], feature_cols).head(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

colors_lr = ["#55A868" if v > 0 else "#C44E52" for v in lr_imp["coefficient"]]
axes[0].barh(lr_imp["feature"][::-1], lr_imp["coefficient"][::-1], color=colors_lr[::-1])
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Logistic Regression — top coefficients")
axes[0].set_xlabel("coefficient (scaled space); green = pushes up, red = pushes down")

axes[1].barh(lgb_imp["feature"][::-1], lgb_imp["gain"][::-1], color="#4C72B0")
axes[1].set_title("LightGBM — top features by gain")
axes[1].set_xlabel("gain")

fig.tight_layout()
plt.show()

## 4. Accuracy by confidence bucket

The whole point of confidence gating: if the model's stated confidence is
meaningful, accuracy should rise as predicted probability moves away from
0.5. Bucket predictions by |P(up) - 0.5| and plot accuracy per bucket
against the base rate — a flat or noisy line means gating buys nothing on
this split, and that finding is reported as-is, not smoothed over.

In [ ]:
def accuracy_by_confidence_bucket(y_true, prob, n_buckets=8):
    distance = np.abs(prob - 0.5)
    edges = np.linspace(0.0, 0.5, n_buckets + 1)
    bucket_idx = np.clip(np.digitize(distance, edges[1:-1]), 0, n_buckets - 1)
    y_pred = (prob >= 0.5).astype(int)
    rows = []
    for b in range(n_buckets):
        mask = bucket_idx == b
        n = int(mask.sum())
        acc = float((y_pred[mask] == y_true[mask]).mean()) if n > 0 else np.nan
        rows.append({
            "bucket": b,
            "range": f"[{edges[b]:.3f}, {edges[b + 1]:.3f})",
            "n": n,
            "accuracy": acc,
        })
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, prob) in zip(axes, MODELS, strict=True):
    table = accuracy_by_confidence_bucket(y_test, prob)
    valid = table.dropna(subset=["accuracy"])
    ax.bar(valid["bucket"], valid["accuracy"], color="#4C72B0")
    for _, row in valid.iterrows():
        ax.text(row["bucket"], row["accuracy"] + 0.01, f"n={row['n']}",
                ha="center", fontsize=8)
    rate, _ = base_rate(y_test)
    ax.axhline(rate, color="gray", linestyle="--", linewidth=1, label="base rate")
    ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8)
    ax.set_xticks(valid["bucket"], valid["range"], rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{name} — accuracy by |P(up) - 0.5| bucket")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
axes[0].set_ylabel("accuracy")
fig.tight_layout()
plt.show()

## 5. Read this honestly

- If accuracy does **not** climb with confidence above, the gating
  threshold in `configs/config.yaml` (`modeling.confidence_threshold`) is
  not doing useful work on this split — it should be revisited rather than
  reported as a feature.
- Bucket counts (`n=`) matter as much as the bars: a high-accuracy bucket
  built on a handful of predictions is noise, not signal.
- None of this substitutes for the leak check in `evaluate.py` — a
  suspiciously clean confidence curve on daily data is itself worth
  re-investigating, not celebrating.